In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

%matplotlib inline

In [2]:
df_2020 = pd.read_csv('/home/jd/ML_Project/data/df_2020_mit_2020_labels.csv')
df_2021 = pd.read_csv('/home/jd/ML_Project/data/df_2021_mit_2021_labels.csv')

In [3]:
df_2020.columns

Index(['system:index', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8',
       'bare_sparse_vegetation', 'built_up', 'cropland', 'grassland',
       'image_count', 'tree_cover', 'water', 'x', 'y', 'year', '.geo',
       'class_label'],
      dtype='str')

In [4]:
df_2020.dtypes

system:index                int64
B11                       float64
B12                       float64
B2                        float64
B3                        float64
B4                        float64
B5                        float64
B6                        float64
B7                        float64
B8                        float64
bare_sparse_vegetation    float64
built_up                  float64
cropland                  float64
grassland                 float64
image_count                 int64
tree_cover                float64
water                     float64
x                         float64
y                         float64
year                        int64
.geo                          str
class_label                 int64
dtype: object

In [5]:
df_2020 = df_2020.drop(columns=['system:index', 'image_count', '.geo', 'year','class_label'])
df_2020 = df_2020.fillna(df_2020.mean())

df_2021 = df_2021.drop(columns=['system:index', 'image_count', '.geo', 'year','class_label'])
df_2021 = df_2021.fillna(df_2021.mean())

In [6]:
# B1, B9, B10

In [7]:
# mean_median_cols = [col for col in df_2020.columns if ('_median' in col)]
# excluded_bands = [col for col in df_2020 if not ('B1_' in col or 'B9_' in col or 'B8A' in col or 'B10_' in col)]

# df_2020['vegetation'] = df_2020[['tree_cover', 'cropland', 'grassland']].sum(axis=1)
# df_2021['vegetation'] = df_2021[['tree_cover', 'cropland', 'grassland']].sum(axis=1)

target_labels = [
    'built_up',
    'tree_cover',
    'grassland',
    'cropland',
    'water'
]

remaining_target = [
    'bare_sparse_vegetation'
]

# sel_col = excluded_bands
# df_2020 = df_2020[sel_col]
# df_2021 = df_2021[sel_col]

In [8]:
df_2020.columns

Index(['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8',
       'bare_sparse_vegetation', 'built_up', 'cropland', 'grassland',
       'tree_cover', 'water', 'x', 'y'],
      dtype='str')

In [9]:
# --- Feature Engineering for df_2020 ---
# Vegetation Indices
df_2020['NDVI'] = (df_2020['B8'] - df_2020['B4']) / (df_2020['B8'] + df_2020['B4'] + 1e-8)
df_2020['EVI'] = 2.5 * ((df_2020['B8'] - df_2020['B4']) / (df_2020['B8'] + 6 * df_2020['B4'] - 7.5 * df_2020['B2'] + 1 + 1e-8))
# df_2020['SAVI'] = ((df_2020['B8'] - df_2020['B4']) * 1.5) / (df_2020['B8'] + df_2020['B4'] + 0.5 + 1e-8)
# df_2020['GNDVI'] = (df_2020['B8'] - df_2020['B3']) / (df_2020['B8'] + df_2020['B3'] + 1e-8)

# Urban & Built-up Indices
df_2020['NDBI'] = (df_2020['B11'] - df_2020['B8']) / (df_2020['B11'] + df_2020['B8'] + 1e-8)

# Water & Moisture Indices
df_2020['NDWI'] = (df_2020['B3'] - df_2020['B8']) / (df_2020['B3'] + df_2020['B8'] + 1e-8)
df_2020['MNDWI'] = (df_2020['B3'] - df_2020['B11']) / (df_2020['B3'] + df_2020['B11'] + 1e-8)


# --- Feature Engineering for df_2021 ---
# Vegetation Indices
df_2021['NDVI'] = (df_2021['B8'] - df_2021['B4']) / (df_2021['B8'] + df_2021['B4'] + 1e-8)
df_2021['EVI'] = 2.5 * ((df_2021['B8'] - df_2021['B4']) / (df_2021['B8'] + 6 * df_2021['B4'] - 7.5 * df_2021['B2'] + 1 + 1e-8))
# df_2021['SAVI'] = ((df_2021['B8'] - df_2021['B4']) * 1.5) / (df_2021['B8'] + df_2021['B4'] + 0.5 + 1e-8)
# df_2021['GNDVI'] = (df_2021['B8'] - df_2021['B3']) / (df_2021['B8'] + df_2021['B3'] + 1e-8)

# Urban & Built-up Indices
df_2021['NDBI'] = (df_2021['B11'] - df_2021['B8']) / (df_2021['B11'] + df_2021['B8'] + 1e-8)

# Water & Moisture Indices
df_2021['NDWI'] = (df_2021['B3'] - df_2021['B8']) / (df_2021['B3'] + df_2021['B8'] + 1e-8)
df_2021['MNDWI'] = (df_2021['B3'] - df_2021['B11']) / (df_2021['B3'] + df_2021['B11'] + 1e-8)

In [10]:
X_train = df_2020.drop(columns=target_labels + remaining_target)
X_train = X_train.drop(columns=['B4', 'B7', 'B8', 'NDBI'])
# X_train = df_2020[['B3_median', 'B8_median', 'B5_median', 'B6_median', 'B4_median', 'B11_median',  'NDVI']]
y_train = df_2020[target_labels]

X_test = df_2021.drop(columns=target_labels + remaining_target)
X_test = X_test.drop(columns=['B4', 'B7', 'B8', 'NDBI'])
# X_test = df_2021[['B3_median', 'B8_median', 'B5_median', 'B6_median', 'B4_median', 'B11_median',  'NDVI']]
y_test = df_2021[target_labels]

In [11]:
X_train.columns, y_train.columns

(Index(['B11', 'B12', 'B2', 'B3', 'B5', 'B6', 'x', 'y', 'NDVI', 'EVI', 'NDWI',
        'MNDWI'],
       dtype='str'),
 Index(['built_up', 'tree_cover', 'grassland', 'cropland', 'water'], dtype='str'))

In [12]:
def evaluate_regression_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_rmse = root_mean_squared_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    test_rmse = root_mean_squared_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"--- {model_name} ---")
    print(f"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}\n")

    return {
        'Model': model_name,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train R2': train_r2,
        'Test R2': test_r2
    }

In [13]:
# from sklearn.multioutput import MultiOutputRegressor
# from sklearn.svm import SVR

# models = {
#     "Linear Regression": Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())]),
#     "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))]),
#     "Lasso Regression": Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(alpha=0.1, max_iter=10000))]),
#     # "SVR": Pipeline([
#     #     ("scaler", StandardScaler()),
#     #     ("svr", MultiOutputRegressor(SVR(kernel='rbf', C=100, epsilon=0.1)))
#     # ]),
#     "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1),
#     "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=865, learning_rate=0.01, random_state=42,
#                                                  max_depth=10,
#                                                  gamma=0.02,
#                                                  min_child_weight=4,
#                                                  reg_alpha=2.098,
#                                                  reg_lambda=1.31,
#                                                  colsample_bytree=0.82,
#                                                  subsample=0.88,
#                                                  n_jobs=-1,
#                                                  ),
#                                     n_jobs=-1
#                                     )
# }

# results = []
# for name, model in models.items():
#     res = evaluate_regression_model(name, model, X_train, X_test, y_train, y_test)
#     results.append(res)


In [14]:
# summary_df = pd.DataFrame(results).sort_values(by='Test RMSE', ascending=True)
# display(summary_df)

In [15]:
# selected_labels = [
#     'tree_cover', 'built_up', 'grassland', 'cropland',
#     'bare_sparse_vegetation', 'water'
# ]

# pd.DataFrame({'mean': df_2020[selected_labels].mean(), 'std': df_2020[selected_labels].std()})

In [16]:
# feature_names = X_train.columns
# importances = models['Random Forest'].feature_importances_

# feature_importance_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Importance': importances
# }).sort_values(by='Importance', ascending=False)

# plt.figure(figsize=(10, 6))
# sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
# plt.title('Top 10 Most Important Features (Random Forest - MultiOutput)')
# plt.show()

# print(feature_importance_df)

In [17]:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# # 1. Use the labels actually used during training (4 labels in your current code)
# current_labels = [
#     # 'vegetation',
#     'built_up',
#     'tree_cover', 'grassland', 'cropland',
#     # 'bare_sparse_vegetation',
#     'water'
# ]

# best_model_name = "XGBoost"
# y_pred_array = models[best_model_name].predict(X_test)

# # 2. Create DataFrame with the matching 4 columns
# y_pred_df = pd.DataFrame(y_pred_array, columns=current_labels)

# # 3. Get dominant classes
# true_dominant = y_test[current_labels].idxmax(axis=1).reset_index(drop=True)
# pred_dominant = y_pred_df.idxmax(axis=1)

# # 4. Build comparison table
# comparison_df = pd.DataFrame({
#     'True_Dominant_Class': true_dominant,
#     'Pred_Dominant_Class': pred_dominant
# })
# comparison_df['Match'] = comparison_df['True_Dominant_Class'] == comparison_df['Pred_Dominant_Class']

# # 5. Output results
# display(comparison_df.head(5))

# # 6. Generate Confusion Matrix
# cm = confusion_matrix(true_dominant, pred_dominant, labels=current_labels)
# plt.figure(figsize=(10, 8))
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=current_labels)
# disp.plot(cmap='Blues', xticks_rotation=45, ax=plt.gca())
# plt.title(f'Confusion Matrix - Dominant Class ({best_model_name})')
# plt.show()

# accuracy = comparison_df['Match'].mean()
# print(f"Overall Dominant Class Accuracy: {accuracy:.4f}")

In [ ]:
# import optuna
# from sklearn.multioutput import MultiOutputRegressor
# from xgboost import XGBRegressor
# from sklearn.metrics import root_mean_squared_error, r2_score

# # Define a callback to print metrics for each trial
# def logging_callback(study, frozen_trial):
#     print(f"Trial {frozen_trial.number} finished with combined score: {frozen_trial.value:.4f}")
#     print(f"Best combined score so far: {study.best_value:.4f}")
#     print("-" * 30)

# def objective(trial):
#     param = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 800),
#         "max_depth": trial.suggest_int("max_depth", 5, 30),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "subsample": trial.suggest_float("subsample", 0.6, 0.9),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
#         "gamma": trial.suggest_float("gamma", 1.0, 10.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 10.0, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
#         "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
#         "n_jobs": -1,
#         "random_state": 42,
#     }

#     model = MultiOutputRegressor(XGBRegressor(**param))
#     model.fit(X_train, y_train)
    
#     # Calculate metrics
#     train_preds = model.predict(X_train)
#     test_preds = model.predict(X_test)
    
#     train_rmse = root_mean_squared_error(y_train, train_preds)
#     test_rmse = root_mean_squared_error(y_test, test_preds)
#     train_r2 = r2_score(y_train, train_preds)
#     test_r2 = r2_score(y_test, test_preds)
    
#     # Calculate the gap for our custom objective
#     gap = abs(test_rmse - train_rmse)
#     combined_score = test_rmse + (0.75 * gap)
    
#     # Print detailed trial metrics immediately
#     print(f"\n[Trial {trial.number}]")
#     print(f"  Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
#     print(f"  Train R2:   {train_r2:.4f} | Test R2:   {test_r2:.4f}")
#     print(f"  Gap:        {gap:.4f}")
#     print(f"  params : {param}")
#     return combined_score

# # Initialize the study
# study = optuna.create_study(direction="minimize")

# # Run optimization with the callback
# study.optimize(objective, n_trials=50, callbacks=[logging_callback])

# # --- Final Results ---
# print("\n" + "="*50)
# print("HYPERPARAMETER TUNING COMPLETE")
# print(f"Best Trial Params: {study.best_params}")
# print("="*50)

# # Final Retrain and Evaluation using your baseline function
# best_model = MultiOutputRegressor(XGBRegressor(**study.best_params, n_jobs=-1, random_state=42))
# evaluate_regression_model("XGBoost_Optuna_Final", best_model, X_train, X_test, y_train, y_test)

/home/jd/ML_Project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-22 00:49:25,883] A new study created in memory with name: no-name-1d9ea191-9cc6-4c54-96b5-1bb0e15c12a0
[I 2026-03-22 00:49:59,113] Trial 0 finished with value: 0.19262292981147766 and parameters: {'n_estimators': 160, 'max_depth': 18, 'learning_rate': 0.01066652836473014, 'subsample': 0.8112640979982169, 'colsample_bytree': 0.6522910470785263, 'gamma': 6.8943236889030635, 'reg_alpha': 0.18566395220923626, 'reg_lambda': 4.620990424504487, 'min_child_weight': 6}. Best is trial 0 with value: 0.19262292981147766.



[Trial 0]
  Train RMSE: 0.1549 | Test RMSE: 0.1764
  Train R2:   0.7411 | Test R2:   0.6843
  Gap:        0.0216
  params : {'n_estimators': 160, 'max_depth': 18, 'learning_rate': 0.01066652836473014, 'subsample': 0.8112640979982169, 'colsample_bytree': 0.6522910470785263, 'gamma': 6.8943236889030635, 'reg_alpha': 0.18566395220923626, 'reg_lambda': 4.620990424504487, 'min_child_weight': 6, 'n_jobs': -1, 'random_state': 42}
Trial 0 finished with combined score: 0.1926
Best combined score so far: 0.1926
------------------------------


[I 2026-03-22 00:50:42,882] Trial 1 finished with value: 0.18577660620212555 and parameters: {'n_estimators': 479, 'max_depth': 14, 'learning_rate': 0.04749495631682532, 'subsample': 0.8569051760965841, 'colsample_bytree': 0.7450591786961398, 'gamma': 8.792976504242166, 'reg_alpha': 1.4936826813774298, 'reg_lambda': 2.2605010298970183, 'min_child_weight': 17}. Best is trial 1 with value: 0.18577660620212555.



[Trial 1]
  Train RMSE: 0.1407 | Test RMSE: 0.1664
  Train R2:   0.7826 | Test R2:   0.7213
  Gap:        0.0258
  params : {'n_estimators': 479, 'max_depth': 14, 'learning_rate': 0.04749495631682532, 'subsample': 0.8569051760965841, 'colsample_bytree': 0.7450591786961398, 'gamma': 8.792976504242166, 'reg_alpha': 1.4936826813774298, 'reg_lambda': 2.2605010298970183, 'min_child_weight': 17, 'n_jobs': -1, 'random_state': 42}
Trial 1 finished with combined score: 0.1858
Best combined score so far: 0.1858
------------------------------


[I 2026-03-22 00:51:26,228] Trial 2 finished with value: 0.18612568825483322 and parameters: {'n_estimators': 203, 'max_depth': 20, 'learning_rate': 0.02367593638320007, 'subsample': 0.7199873931311646, 'colsample_bytree': 0.7669875146935242, 'gamma': 7.437286334588043, 'reg_alpha': 1.7192605316995633, 'reg_lambda': 0.33422214500132036, 'min_child_weight': 6}. Best is trial 1 with value: 0.18577660620212555.


...

[I 2026-03-22 01:45:26,945] Trial 48 finished with value: 0.18235984817147255 and parameters: {'n_estimators': 556, 'max_depth': 10, 'learning_rate': 0.04511497669216503, 'subsample': 0.7076043731604057, 'colsample_bytree': 0.6971251397212214, 'gamma': 1.1358099436059712, 'reg_alpha': 6.412824829112223, 'reg_lambda': 0.759041133046522, 'min_child_weight': 11}. Best is trial 48 with value: 0.18235984817147255.



[Trial 48]
  Train RMSE: 0.1301 | Test RMSE: 0.1599
  Train R2:   0.8159 | Test R2:   0.7436
  Gap:        0.0299
  params : {'n_estimators': 556, 'max_depth': 10, 'learning_rate': 0.04511497669216503, 'subsample': 0.7076043731604057, 'colsample_bytree': 0.6971251397212214, 'gamma': 1.1358099436059712, 'reg_alpha': 6.412824829112223, 'reg_lambda': 0.759041133046522, 'min_child_weight': 11, 'n_jobs': -1, 'random_state': 42}
Trial 48 finished with combined score: 0.1824
Best combined score so far: 0.1824
------------------------------


[I 2026-03-22 01:47:56,592] Trial 49 finished with value: 0.1821778044104576 and parameters: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.04475778468557638, 'subsample': 0.6876719545127158, 'colsample_bytree': 0.6983814307724542, 'gamma': 1.0051560079684305, 'reg_alpha': 6.714222578424157, 'reg_lambda': 0.6964390261386526, 'min_child_weight': 11}. Best is trial 49 with value: 0.1821778044104576.



[Trial 49]
  Train RMSE: 0.1296 | Test RMSE: 0.1597
  Train R2:   0.8172 | Test R2:   0.7447
  Gap:        0.0300
  params : {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.04475778468557638, 'subsample': 0.6876719545127158, 'colsample_bytree': 0.6983814307724542, 'gamma': 1.0051560079684305, 'reg_alpha': 6.714222578424157, 'reg_lambda': 0.6964390261386526, 'min_child_weight': 11, 'n_jobs': -1, 'random_state': 42}
Trial 49 finished with combined score: 0.1822
Best combined score so far: 0.1822
------------------------------

HYPERPARAMETER TUNING COMPLETE
Best Trial Params: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.04475778468557638, 'subsample': 0.6876719545127158, 'colsample_bytree': 0.6983814307724542, 'gamma': 1.0051560079684305, 'reg_alpha': 6.714222578424157, 'reg_lambda': 0.6964390261386526, 'min_child_weight': 11}
--- XGBoost_Optuna_Final ---
Train R2: 0.8172 | Test R2: 0.7447
Train RMSE: 0.1296 | Test RMSE: 0.1597



{'Model': 'XGBoost_Optuna_Final',
 'Train RMSE': 0.12963785231113434,
 'Test RMSE': 0.1596606820821762,
 'Train R2': 0.8172420263290405,
 'Test R2': 0.7446914911270142}

In [20]:
import optuna
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

# Define a callback to print metrics for each trial
def logging_callback(study, frozen_trial):
    print(f"Trial {frozen_trial.number} finished with combined score: {frozen_trial.value:.4f}")
    print(f"Best combined score so far: {study.best_value:.4f}")
    print("-" * 30)

def objective(trial):
    # Hyperparameters specific to Random Forest
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.6, 0.9),
        "n_jobs": -1,
        "random_state": 42,
    }

    # Model changed to RandomForestRegressor
    model = MultiOutputRegressor(RandomForestRegressor(**param))
    model.fit(X_train, y_train)
    
    # Calculate metrics
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    train_rmse = root_mean_squared_error(y_train, train_preds)
    test_rmse = root_mean_squared_error(y_test, test_preds)
    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)
    
    # Calculate the gap for our custom objective
    gap = abs(test_rmse - train_rmse)
    combined_score = test_rmse + (0.75 * gap)
    
    # Print detailed trial metrics immediately
    print(f"\n[Trial {trial.number}]")
    print(f"  Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"  Train R2:   {train_r2:.4f} | Test R2:   {test_r2:.4f}")
    print(f"  Gap:        {gap:.4f}")
    print(f"  params : {param}")
    return combined_score

# Initialize the study
study = optuna.create_study(direction="minimize")

# Run optimization with the callback
study.optimize(objective, n_trials=50, callbacks=[logging_callback])

# --- Final Results ---
print("\n" + "="*50)
print("HYPERPARAMETER TUNING COMPLETE")
print(f"Best Trial Params: {study.best_params}")
print("="*50)

# Final Retrain and Evaluation using your baseline function
best_model = MultiOutputRegressor(RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42))
evaluate_regression_model("RFR_Optuna_Final", best_model, X_train, X_test, y_train, y_test)

[I 2026-03-22 02:11:10,029] A new study created in memory with name: no-name-8da0f9b5-e22d-4d60-8433-112f03f30f0c
[I 2026-03-22 02:26:49,892] Trial 0 finished with value: 0.1904882616148547 and parameters: {'n_estimators': 262, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 0.6646313117695952}. Best is trial 0 with value: 0.1904882616148547.



[Trial 0]
  Train RMSE: 0.1340 | Test RMSE: 0.1663
  Train R2:   0.8131 | Test R2:   0.7249
  Gap:        0.0323
  params : {'n_estimators': 262, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 0.6646313117695952, 'n_jobs': -1, 'random_state': 42}
Trial 0 finished with combined score: 0.1905
Best combined score so far: 0.1905
------------------------------


[I 2026-03-22 02:46:58,644] Trial 1 finished with value: 0.2087558970808792 and parameters: {'n_estimators': 207, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 0.7002282171569837}. Best is trial 0 with value: 0.1904882616148547.



[Trial 1]
  Train RMSE: 0.0918 | Test RMSE: 0.1586
  Train R2:   0.9117 | Test R2:   0.7486
  Gap:        0.0668
  params : {'n_estimators': 207, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 0.7002282171569837, 'n_jobs': -1, 'random_state': 42}
Trial 1 finished with combined score: 0.2088
Best combined score so far: 0.1905
------------------------------


[I 2026-03-22 03:47:37,087] Trial 2 finished with value: 0.19483527829057642 and parameters: {'n_estimators': 634, 'max_depth': 15, 'min_samples_split': 19, 'min_samples_leaf': 10, 'max_features': 0.7911795938757374}. Best is trial 0 with value: 0.1904882616148547.


...


[Trial 12]
  Train RMSE: 0.0953 | Test RMSE: 0.1581
  Train R2:   0.9033 | Test R2:   0.7501
  Gap:        0.0628
  params : {'n_estimators': 484, 'max_depth': 21, 'min_samples_split': 16, 'min_samples_leaf': 8, 'max_features': 0.8254380296436806, 'n_jobs': -1, 'random_state': 42}
Trial 12 finished with combined score: 0.2052
Best combined score so far: 0.1897
------------------------------


[I 2026-03-22 09:50:58,351] Trial 13 finished with value: 0.18999231098264485 and parameters: {'n_estimators': 327, 'max_depth': 11, 'min_samples_split': 16, 'min_samples_leaf': 8, 'max_features': 0.7128113610484731}. Best is trial 7 with value: 0.18972697440055214.



[Trial 13]
  Train RMSE: 0.1291 | Test RMSE: 0.1639
  Train R2:   0.8260 | Test R2:   0.7326
  Gap:        0.0348
  params : {'n_estimators': 327, 'max_depth': 11, 'min_samples_split': 16, 'min_samples_leaf': 8, 'max_features': 0.7128113610484731, 'n_jobs': -1, 'random_state': 42}
Trial 13 finished with combined score: 0.1900
Best combined score so far: 0.1897
------------------------------


[W 2026-03-22 10:10:32,183] Trial 14 failed with parameters: {'n_estimators': 500, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 0.8885776714166652} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/jd/ML_Project/.venv/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_179560/1301842602.py", line 30, in objective
    test_preds = model.predict(X_test)
  File "/home/jd/ML_Project/.venv/lib/python3.14/site-packages/sklearn/multioutput.py", line 306, in predict
    y = Parallel(n_jobs=self.n_jobs)(
        delayed(e.predict)(X) for e in self.estimators_
    )
  File "/home/jd/ML_Project/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py", line 91, in __call__
    return super().__call__(iterable_with_config_and_warning_filters)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/hom

KeyboardInterrupt: 